# Experiment 7: response recruitment and containment

Stage 1 of the overlap/selectivity workflow: quantify recruitment, signed response-window activity, and containment. Every figure places **left stimuli on the left and right stimuli on the right**, in the same functional order and with matching scales. Later similarity and selectivity analyses are outside this first version.

This notebook runs independently of notebook 01. It reuses its cohort, portable loading, timing, and overlap/count activity criteria. Existing scientific helper behavior is unchanged. All calculations use the same kept-neuron identities within each fish; fish contribute equally to displayed means.

The labels R300, R600, RF600, F300, F600, and DF300 denote **stimulus variants**. Interpret the plots as response recruitment and containment, without assuming an anatomical hierarchy or a frequency-only manipulation.


## 1. Setup

Find the repository and import the shared loading, analysis, figure, and reporting owners.


In [ ]:
from pathlib import Path
import sys
import importlib
from IPython.display import display
import matplotlib.pyplot as plt

repo_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'src').is_dir())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import src.several_fish_loading as several_loading
import src.several_fish_diagnostics as several_diagnostics
import src.several_fish_figures as several_figures
import src.several_fish_reporting as several_reporting
for module in (several_loading, several_diagnostics, several_figures, several_reporting):
    importlib.reload(module)


## 2. Cohort, mirrored stimulus groups, and activity criteria

Keep experiment-specific choices here. The activity criteria match notebook 01's overlap/count figures. `data_base=None` discovers OneDrive automatically; a manual override is the parent of the university OneDrive folder.


In [ ]:
SIDE_GROUPS = {
    'left': {
        'control': 'LeBcontrol',
        'rocking': ['R300Le', 'R600Le', 'RF600Le'],
        'flicker': ['F300Le', 'F600Le', 'DF300Le'],
    },
    'right': {
        'control': 'RiBcontrol',
        'rocking': ['R300Ri', 'R600Ri', 'RF600Ri'],
        'flicker': ['F300Ri', 'F600Ri', 'DF300Ri'],
    },
}
CONDITION_LABELS = ['Control', 'R300', 'R600', 'RF600', 'F300', 'F600', 'DF300']
USER_SETTINGS = {
    'experiment_name': 'Exp_7_trans_stat',
    'analysis_label': 'Exp 7 recruitment and containment',
    'data_base': None,
    'fish_ids': ['L742_f03', 'L742_f05', 'L742_f06', 'L742_f09', 'L742_f10'],
    'selected_blocks': ['B1', 'B2', 'B3', 'B4'],
    'verbose_loading': False,
    'timing': {'fps_2p': 2.0, 't_pre_s': 5.0, 't_post_s': 25.0,
               'motion_onset_s': 8.0, 'tau_s': 6.0},
    'stim_order': [7, 8, 10, 12, 3, 5, 1, 14, 9, 11, 13, 4, 6, 2],
}
ACTIVE_SETTINGS = {
    'active_fraction_threshold': 0.10,
    'min_epoch_s': 1.0,
    'min_active_reps': 2,
    'expected_reps': 4,
    'require_expected_reps': True,
}
REPORT_SETTINGS = {
    'save_report': False,
    'run_label': 'exp7_recruitment_containment',
    'report_format': 'html',
    'export_notebook': True,
    'fallback_to_html': True,
    'comments': 'Stage 1; enable saving only after reviewing both sides.',
}


## 3. Independent loading and preflight

Load all five fish, validate the configured stimuli, and show the selected roots. The analysis step also validates the named groups and filtered-raster/continuous-trace row contract for every fish.


In [ ]:
raster_inputs = several_loading.load_and_preflight_fish_raster_inputs(USER_SETTINGS)
globals().update(raster_inputs)
print('Data root:', main_path)
print('Analysis root:', analysis_path)
print(f'Loaded {len(fish_ids)} fish.')


## 4. Build aligned analysis inputs once

Recruitment uses the canonical significant-raster activity decision. Continuous activity uses signed z-score AUC across the **same kept neurons**, including inactive neurons, divided by the actual trapezoidal integration span. This is response-window z-score activity, with no additional baseline subtraction or clipping at zero.

The existing timing helper starts each window 8 s after stimulus onset and requests stimulus motion duration plus `2 * tau_s`. It clips to the available aligned trace. The table below makes requested and observed durations explicit; conclusions concern these observed windows. Raw AUC, sample bounds, individual-neuron responses, and fish identities remain in `analysis['tables']`.


In [ ]:
analysis = several_diagnostics.build_recruitment_containment_analysis(
    all_fish_data, fish_ids, SIDE_GROUPS, timing, ACTIVE_SETTINGS, CONDITION_LABELS,
)
display(analysis['tables']['window_summary'])


## 5. Recruitment and response-window activity

Top: fraction of kept neurons active for each stimulus. Bottom: mean normalized AUC across that same kept-neuron population. Colored points/lines identify fish; black diamonds show the equally weighted fish mean. Both sides share the corresponding y scale. The activity panel combines recruitment and response magnitude; it does not establish stronger activity specifically within shared active neurons.


In [ ]:
FIGSIZE = (14, 9)
fig_recruitment, axes_recruitment = several_figures.plot_recruitment_activity_summary(
    analysis, figsize=FIGSIZE,
)
plt.show()


## 6. Directional containment and Jaccard overlap

Top: row A / column B is `|A intersection B| / |A|`, the fraction of A-responsive neurons also active to B. Bottom: Jaccard is intersection divided by union. Matrices are means of per-fish fractions, without weighting fish by neuron count. All heatmaps use the same 0-1 color scale. Gray cells are undefined; empty populations are not reported as zero overlap. Individual fish matrices and valid-fish counts remain available in the result tables.


In [ ]:
FIGSIZE = (14, 12)
HEATMAP_CMAP = 'viridis'
fig_overlap, axes_overlap = several_figures.plot_recruitment_containment_matrices(
    analysis, figsize=FIGSIZE, cmap=HEATMAP_CMAP,
)
plt.show()


## 7. Containment in control and in all rocking variants

Top: fraction of each weaker-stimulus population also responding to its side's control. Bottom: fraction of each flicker population responding to **every rocking variant**, using their intersection, not their union. The denominator is the source stimulus's active population. Empty sources leave gaps. Non-contained counts and fractions are retained in `analysis['tables']['containment_summary']`.


In [ ]:
FIGSIZE = (14, 9)
fig_containment, axes_containment = several_figures.plot_recruitment_containment_summary(
    analysis, figsize=FIGSIZE,
)
plt.show()


## 8. Review before extending the analysis

Check whether greater control recruitment and activity recur across individual fish and on both sides. Use the directional summaries to quantify containment and preserve the exceptions. The plotted overlaps describe activity decisions under the configured criteria; they do not by themselves establish identical coding, dedicated neuron types, or an anatomical hierarchy.

Further questions about diversity within rocking and flicker will be addressed after reviewing this stage, using continuous response similarity, response breadth, sparseness, and independent-trial preference validation.


## 9. Optional reviewed-run report

Saving is disabled by default. Save the notebook with current outputs before enabling the existing report exporter.


In [ ]:
notebook_path = repo_root / 'notebooks' / 'calcium' / 'exp_07_trans_stat' / '02_response_overlap_and_selectivity.ipynb'
report_result = several_reporting.save_analysis_report_run(
    settings=settings, report_settings=REPORT_SETTINGS, analysis_path=analysis_path,
    tables=analysis['tables'], notebook_path=notebook_path,
    report_name='02_response_overlap_and_selectivity_report',
    extra_metadata={
        'side_groups': SIDE_GROUPS, 'condition_labels': CONDITION_LABELS,
        'active_settings': ACTIVE_SETTINGS, 'data_root': main_path,
        'analysis_root': analysis_path, 'aggregation': 'equal_fish_weight',
    },
)
